In [0]:
# Notebook: 05_scd2_versioning.py
# Purpose : Apply SCD Type 2 logic to Silver Patient table (pattern reusable for all)

from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, sha2, concat_ws, when
)
from config.config import SILVER_DB
import requests, uuid
from datetime import datetime
from importlib import reload
import sys


# Add repository root to Python path for module imports
repo_root = "/Workspace/Users/vyshnavi.thunuguntla@outlook.com/databricks-repo-FHIR"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import and reload to pick up latest changes
import audit_utils
reload(audit_utils)
from audit_utils import audit_start, audit_end

spark = SparkSession.builder.getOrCreate()


def apply_scd2(target_table: str, source_df, pk_col: str, tracked_cols: list):
    """
    Generic SCD Type 2 merge.
    Adds:  effective_start, effective_end, is_current, row_hash
    """
    run_id = str(uuid.uuid4()) 
    full_table = f"{SILVER_DB}.{target_table}_scd2"
    
    eff_st = current_timestamp()
    # Hash tracked columns to detect changes
    source_df = source_df.withColumn(
        "row_hash",
        sha2(concat_ws("||", *[col(c).cast("string") for c in tracked_cols]), 256),
    ).withColumn("effective_start", eff_st) \
     .withColumn("effective_end",   lit(None).cast("timestamp")) \
     .withColumn("is_current",      lit(True))
    
    # Create on first run
    if not spark.catalog.tableExists(full_table):
        source_df.write.format("delta").saveAsTable(full_table)
        print(f"[SCD2] Created {full_table}")
        return
    
    CNT1 = spark.sql(f"SELECT COUNT(*) FROM {full_table} ").collect()[0][0]
    started_at = datetime.utcnow() # current_timestamp(),
    ########
    audit_start(
        run_id        = run_id,
        stage         = "SILVER_SCD2",
        resource_type = f"{target_table}_scd2",
        operation     = "INGEST",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "02_silver_ingest_scd2"
    )

    target = DeltaTable.forName(spark, full_table)

    # Step 1: Expire changed rows in target
    (
        target.alias("tgt")
        .merge(
            source_df.alias("src"),
            f"tgt.{pk_col} = src.{pk_col} AND tgt.is_current = true AND tgt.row_hash != src.row_hash",
        )
        .whenMatchedUpdate(set={
            "effective_end": "src.effective_start",
            "is_current":    "false",
        })
        .execute()
    )

    # Step 2: Insert new / changed rows
    existing = spark.table(full_table).filter(col("is_current") == True).select(pk_col, "row_hash")
    new_rows  = source_df.join(existing, on=pk_col, how="left_anti")   # brand-new
    changed   = source_df.join(
        existing.withColumnRenamed("row_hash", "old_hash"),
        on=pk_col, how="inner"
    ).filter(col("row_hash") != col("old_hash")).drop("old_hash")

    to_insert = new_rows.union(changed)
    if to_insert.count() > 0:
        to_insert.write.format("delta").mode("append").saveAsTable(full_table)

    print(f"[SCD2] {full_table}: {to_insert.count()} rows inserted/updated")
    CNT2 = spark.sql(f"SELECT COUNT(*) FROM {full_table} ").collect()[0][0]
    upd_cnt = spark.sql(f"SELECT COUNT(*) FROM {full_table} WHERE effective_end IS NOT NULL AND is_current = false").collect()[0][0]
    ins_cnt = spark.sql(f"SELECT COUNT(*) FROM {full_table} WHERE is_current = true").collect()[0][0]
    audit_end(
        run_id        = run_id,
        stage         = "SILVER_SCD2",
        resource_type = f"{target_table}_scd2",
        operation     = "INGEST",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
        updated_count= upd_cnt ,
        inserted_count= ins_cnt 
    )

# ──  Apply to Patient ─────────────────────────────────────────────────
patient_df = spark.table(f"{SILVER_DB}.patient")
apply_scd2(
    target_table = "patient",
    source_df    = patient_df,
    pk_col       = "patient_id",
    tracked_cols = ["gender", "birth_date", "family_name", "given_name"],
)
# ──  Apply to encounter ─────────────────────────────────────────────────
encounter_df = spark.table(f"{SILVER_DB}.encounter")
apply_scd2(
    target_table = "encounter",
    source_df    = encounter_df,
    pk_col       = "encounter_id",
    tracked_cols = ["status", "encounter_class", "patient_ref", "period_start", "period_end", "patient_id"]
)
# ──  Apply to observation ─────────────────────────────────────────────────
observation_df = spark.table(f"{SILVER_DB}.observation")
apply_scd2(
    target_table = "observation",
    source_df    = observation_df,
    pk_col       = "observation_id",
    tracked_cols = ["status" ,"patient_ref" ,"encounter_ref" ,"effective_datetime" ,"loinc_code" ,"observation_name" ,"value_quantity" ,"value_unit","patient_id" ,"encounter_id"]
)

# ──  Apply to condition ─────────────────────────────────────────────────
condition_df = spark.table(f"{SILVER_DB}.condition")
apply_scd2(
    target_table = "condition",
    source_df    = condition_df,
    pk_col       = "condition_id",
    tracked_cols = ["clinical_status" ,"patient_ref" ,"encounter_ref" ,"snomed_code" ,"condition_name" ,"onset_datetime" ,"patient_id" ,"encounter_id"]
)

print("✅ SCD Type 2 versioning complete.")